In [6]:
from triplet_extraction.src.doc_extraction.parse_text_to_section import is_marker_line
import re

CHUONG_PATTERN = re.compile(r"^chương\s+(?:[ivxlcdm]+|\d+)", re.IGNORECASE)
MUC_PATTERN    = re.compile(r"^mục\s+(?:[ivxlcdm]+|\d+)", re.IGNORECASE)
DIEU_PATTERN   = re.compile(r"^điều\s+(?:[ivxlcdm]+|\d+)", re.IGNORECASE)

def merge_chuong_muc_blocks(text: str) -> str:
    if not isinstance(text, str):
        return text

    lines = text.splitlines()
    result_lines = []

    i = 0
    n = len(lines)

    while i < n:
        line = lines[i].strip()

        # Gặp Chương hoặc Mục
        if CHUONG_PATTERN.match(line) or MUC_PATTERN.match(line) or DIEU_PATTERN.match(line):
            merged_parts = [line]
            i += 1

            # Collect cho đến khi gặp marker khác
            while i < n:
                next_line = lines[i].strip()

                if is_marker_line(next_line):
                    break

                if next_line:
                    merged_parts.append(next_line)
                i += 1

            # Gộp thành 1 dòng, thay \n bằng " "
            merged_line = " ".join(merged_parts)
            result_lines.append(merged_line)
        else:
            result_lines.append(lines[i])
            i += 1

    return "\n".join(result_lines)

In [7]:
from pathlib import Path
import pandas as pd

data_folder = Path(r"E:\Github\LawAssistant\triplet_extraction\data\luat_dat_dai")
extracted_csv = data_folder / "extracted_texts_google_fixed.csv"

df = pd.read_csv(extracted_csv, encoding="utf-8-sig")
print(f"Loaded {len(df)} documents from CSV")

df["combined_text"] = df["combined_text"].apply(merge_chuong_muc_blocks)
fixed_csv = data_folder / "extracted_texts_google_fixed_v1.csv"
df.to_csv(fixed_csv, index=False, encoding="utf-8-sig")

Loaded 61 documents from CSV
